# 01 - Collect a Training Dataset

This notebook shows the standard Mouse Core data collection pattern:

1. Build one or more environments with `mouse_gym.EnvConfig` and `make_group_env`.
2. Create one `Datastore` for each environment stream.
3. Step the grouped environment, merge each input and output into a row, and append rows to the matching store.
4. Publish the stores with `push_stores_to_hub` so training notebooks can load the same data later.

The environment used here is Procedural Frozen Lake, but the Mouse Core pattern is environment-agnostic. Any environment output that can be represented as dictionaries of Python values can be written to a `Datastore`.


In [ ]:
import torch
import numpy as np

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_group_env
from mouse_core.data import Datastore, push_stores_to_hub


DATASET_ID = "mouse-example-dataset"  # Hugging Face dataset repo for push_stores_to_hub
NUM_ENVS = 30                         # number of environment streams in the GroupEnv
STEPS_PER_ENV = 50_000                # rows collected for each datastore
MAX_EPISODE_STEPS = 30                # environment-specific episode limit
EPISODES_PER_TASK = 20                # environment-specific task length
ORACLE_PROB_END = 1.0                 # final probability of using the expert action during collection

# Total rows collected are NUM_ENVS * STEPS_PER_ENV, split across one datastore per env.


## Build Environment

`EnvConfig` is the bridge between an environment package and Mouse Core data collection. Each config names an environment, gives it a stable stream name, and passes reset/task options plus environment-specific keyword arguments.

`make_group_env(configs)` returns a `GroupEnv`. A group environment lets you step many independent environment instances with one call. The returned `env.names` are useful because they line up with outputs, metrics, and the datastores created below.

A few config fields are worth noticing when adapting this example:

- `name` becomes the stream name you can reuse for a matching `Datastore`.
- `reset_seed` and environment seeds make generated data reproducible.
- `episodes_per_task` and `task_reset_options` define where task boundaries occur. Mouse Core objectives read those boundaries from the `done` field later.
- `kwargs` are passed through to the underlying Gymnasium environment. Mouse Core does not require these specific fields; they are specific to this example environment.


In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_{i}",
        reset_seed=i,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},  # forwarded to the environment at task reset
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i,
            "slippery_success_rate": 1.0,  # environment-specific option
            "permute_obs": True,      # environment-specific option
            "permute_actions": True,  # environment-specific option
            "emit_q_star": True,      # include a field we flatten for downstream objectives
        },
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)

## Create Datastores

A `Datastore` is an append-only sequence of step dictionaries backed by Hugging Face Arrow data. It does not require a fixed schema up front; the rows you append define the columns.

The common pattern is one datastore per environment stream. Keeping streams separate preserves sequential ordering inside each environment while still letting `DataLoader` mix streams during training.


In [ ]:
stores = [Datastore(name=name) for name in env.names]  # one ordered store per env stream

## Collect Transitions (`run_rollout`)

Each environment step has two parts:

1. The input you sent, such as `{"action": ...}`.
2. The output returned by the environment, such as `observation`, `reward`, `done`, indices, and optional `info` values.

Mouse Core training examples expect a flat row dictionary, so this notebook merges input and output into one row before appending it. If an environment returns nested `info`, flatten only the fields your objective or model will need later.

The first `env.step(...)` also performs the initial reset for each environment instance. Storing that row keeps every stream aligned from its first observed state.


In [ ]:
def choose_inputs(*, outputs, env, oracle_prob: float) -> list[dict]:
    """Build one input dictionary per environment output."""
    random_inputs = env.sample_random_input()
    inputs = []
    for i, out in enumerate(outputs):
        if torch.rand(1).item() >= oracle_prob:
            inputs.append(random_inputs[i])
        else:
            action = out['info']['q_star'].argmax()
            inputs.append({'action': np.int64(action)})
    return inputs
if not 0.0 <= ORACLE_PROB_END <= 1.0:
    raise ValueError(f'ORACLE_PROB_END must be in [0, 1], got {ORACLE_PROB_END}.')

def run_rollout(*, env, stores: list, steps_per_env: int) -> None:
    """Step environments and append flat rows into per-stream datastores."""
    outputs = None
    env.metrics.clear()
    for step in range(steps_per_env):
        oracle_prob = ORACLE_PROB_END * (step / (steps_per_env - 1))
        if outputs is None:
            inputs = env.sample_random_input()
        else:
            inputs = choose_inputs(outputs=outputs, env=env, oracle_prob=oracle_prob)
        outputs = env.step(inputs)
        for store, inp, out in zip(stores, inputs, outputs):
            row = {**inp, **out}
            info = row.pop('info')
            row['info_q_star'] = info['q_star']
            store.append(row)
        if (step + 1) % 1000 == 0:
            task_scores = [r for env_tasks in env.metrics.task_cum_rewards for r in env_tasks]
            mean_task_score = sum(task_scores) / len(task_scores) if task_scores else float('nan')
            print(f'step {step + 1}/{steps_per_env} | oracle_prob={oracle_prob:.2f} | {len(task_scores)} tasks | mean task score {mean_task_score:.2f}/{EPISODES_PER_TASK}')
            env.metrics.clear()
run_rollout(env=env, stores=stores, steps_per_env=STEPS_PER_ENV)
env.close()


## Push to the Hub

`push_stores_to_hub` uploads all datastores to one Hugging Face dataset repository. Each datastore is saved as a named config, so downstream code can reload the same independent streams with `load_stores_from_hub`.


In [ ]:
url = push_stores_to_hub(stores=stores, repo_id=DATASET_ID, split='train', private=False)
print(f"Pushed to {url}")